In [ ]:
# ============================================================
# 第 5 章 Part-4：用本地 Llama-3-8B-Instruct 作为「LLM 评委」评测指令回答
# ------------------------------------------------------------
# 方法论 = LLM-as-a-judge：让一个较强的 LLM 对模型回答按 0-100 打分，
# 从而对比微调前 (response_before) 与微调后 (response_after) 的平均分。
# 优点：比 MMLU 选择题更贴近开放式生成质量；缺点：评分带主观性、依赖评委模型能力。
# ============================================================
#Instruction finetuning (part 4; evaluating instruction responses locally using a Llama 3 model)

In [ ]:
# 打印所需依赖的版本号，便于复现环境。
from importlib.metadata import version

pkgs = ["tqdm",    # Progress bar
        ]

for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
# --------- 加载 JSON 数据 ---------
# Load JSON Entries

In [ ]:
# 加载预先准备好的对比数据：每条样本同时含微调前/后两种回答，供评委打分。
import json

# 该文件每条 dict 应包含：instruction / input / output（标准答案）
# 以及 response_before（微调前回答）、response_after（微调后回答）。
json_file = "test_response_before_after.json"

with open(json_file, "r") as file:
    json_data = json.load(file)

print("Number of entries:", len(json_data))

In [ ]:
# 查看第 0 条样本的完整字段结构，确认包含 response_before / response_after。
json_data[0]

In [ ]:
# 复用 format_input 构造统一提示模板（与前几个 part 一致），用于拼接给评委的上下文。
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. Write a response that "
        f"appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    # ✅ 已修复：删除了原来一条孤立无效语句 `instruction_text + input_text`（无赋值/return 的笔误）。
    return instruction_text + input_text

print(format_input(json_data[0])) # input

In [ ]:
# 对比查看「标准答案 output」与「微调前回答 response_before」。
# ✅ 已修复：原代码用裸表达式 json_data[0]["output"]（不会显示，Notebook 只显示最后一个表达式）；
#    现改用 print() 同时打印两者。
print(json_data[0]["output"])
print(json_data[0]["response_before"])

In [ ]:
# 加载本地 Llama-3-8B-Instruct 作为评委模型 (judge)。
# 选择一个指令对齐、能力较强的模型当评委，是 LLM-as-a-judge 的关键前提。
from litgpt import LLM

llm = LLM.load("meta-llama/Meta-Llama-3-8B-Instruct")

In [ ]:
# 定义打分函数：对指定回答字段 (json_key) 逐条让评委打 0-100 分。
from tqdm import tqdm


def generate_model_scores(json_data, json_key):
    scores = []
    for entry in tqdm(json_data, desc="Scoring entries"):
        # 评分提示：给出「输入(含指令) + 标准答案 output + 待评回答」，要求只输出 0-100 的整数。
        # 提供标准答案作为参照，可让评委的评分更有依据（reference-based scoring）。
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"score the model response `{entry[json_key]}`"
            f" on a scale from 0 to 100, where 100 is the best score. "
            f"Respond with the integer number only."
        )
        # max_new_tokens=50 足够输出一个整数；限制长度可防止评委长篇大论。
        score = llm.generate(prompt, max_new_tokens=50)
        try:
            # 尝试把评委输出解析为整数；解析失败（非纯数字）则跳过该条。
            scores.append(int(score))
        except ValueError:
            continue

    return scores

In [ ]:
# --------- 评测两组回答 ---------
#Evaluate the LLMs

In [ ]:
# 分别对「微调前」和「微调后」的回答打分，并打印各自的有效样本数与平均分。
# 平均分升高即说明指令微调带来了可量化的质量提升。
for model in ("response_before", "response_after"):

    scores = generate_model_scores(json_data, model)
    print(f"\n{model}")
    # 有效评分数可能少于总数（部分回答评委未给出可解析的整数，被跳过）。
    print(f"Number of scores: {len(scores)} of {len(json_data)}")
    # 求平均分作为该组回答的总体质量指标。
    print(f"Average score: {sum(scores)/len(scores):.2f}\n")